In [1]:
from dotenv import load_dotenv
import os

from cognichem_client import CogniChemClient


# Load API key from .env file in current working directory
load_dotenv()
API_KEY = os.getenv("API_KEY")

# Initialize the CogniChem client
client = CogniChemClient(API_KEY)

In [ ]:
import pandas as pd


df = pd.read_csv("data/freesolv_database.csv")
SMILES = df["smiles"].tolist()
TARGETS = df["targets"].tolist()

len(SMILES), len(TARGETS)

(642, 642)

In [3]:
import time


MODEL_NAME = f"freesolv-mpnn-{int(time.time())}"

process_id = client.jobs.run(
    job_name=f"Train model: {MODEL_NAME}",
    job_type="train-mpnn",
    payload={
        "model_name": MODEL_NAME,
        "input_data": SMILES,
        "input_format": "smiles",
        "targets": [float(t) for t in TARGETS],
        "unit": "kcal/mol",
        "model_description": "MPNN model trained on FreeSolv dataset",
        "tags": ["freesolv", "mpnn", "kcal/mol"],
        "model_params": {
            "featureset": "organic",
            "generate_3d": True,
            "add_hydrogens": True,
            "validation_size": 0.2,
            "random_seed": 42,
            "mp_hidden_dim": 128,
            "mp_depth": 3,
            "n_ffn_layers": 2,
            "ffn_hidden_dim": 128,
            "learning_rate": 0.0001,
            "n_epochs": 50,
            "batch_size": 32,
        }
    },
    get_result=False
)["process_id"]

print(f"Training job completed. Process ID: {process_id}")

Training job completed. Process ID: fc-01KGNW8V5YY9M9RN8SD36REF3H


In [4]:
client.jobs.status(process_id)

{'process_id': 'fc-01KGNW8V5YY9M9RN8SD36REF3H',
 'status': 'completed',
 'message': 'Process finished'}

In [5]:
client.inference.get_model(MODEL_NAME)

{'model_names': None,
 'model_info': {'name': 'freesolv-mpnn-1770260816',
  'description': 'MPNN model trained on FreeSolv dataset',
  'id': 'c1qrphm5',
  'state': 'finished',
  'created_at': '2026-02-05T03:07:04Z',
  'config': {'unit': 'kcal/mol',
   'mp_depth': 3,
   'n_epochs': 50,
   'batch_size': 32,
   'featureset': 'organic',
   'model_name': 'freesolv-mpnn-1770260816',
   'generate_3d': True,
   'random_seed': 42,
   'input_format': 'smiles',
   'n_ffn_layers': 2,
   'add_hydrogens': True,
   'learning_rate': 0.0001,
   'mp_hidden_dim': 128,
   'ffn_hidden_dim': 128,
   'validation_size': 0.2,
   'remove_hydrogens': False},
  'history': [{'_step': 0,
    'invalid_molecules': [],
    '_runtime': 9.172402462,
    'total_time': 9.066989421844482,
    'featurize_time': 9.06698751449585,
    '_timestamp': 1770260833.4098444},
   {'_runtime': 9.328278481,
    'val/rmse': 1.2439552545547485,
    'val/loss': 1.5474247932434082,
    'val/mae': 0.9683017730712891,
    'val/r2': -0.058414